# 🚗 Project 1: Used Car Price Predictor
### Dataset: CarDekho / Vehicle Dataset
---

## 📋 Dataset Overview & Expected Target

| Feature | Description | Type |
|---------|-------------|------|
| `name` / `brand` | Car manufacturer and model name | Categorical |
| `year` | Year of manufacture | Numerical → age proxy |
| `km_driven` | Total kilometres driven | Numerical |
| `fuel` | Fuel type: Petrol / Diesel / CNG / LPG | Categorical |
| `seller_type` | Individual / Dealer / Trustmark | Categorical |
| `transmission` | Manual / Automatic | Categorical |
| `owner` | 1st / 2nd / 3rd / 4th+ owner | Ordinal Categorical |
| `mileage` | Fuel efficiency (km/l or km/kg) | Numerical (string-encoded) |
| `engine` | Engine displacement (CC) | Numerical (string-encoded) |
| `max_power` | Power output (bhp) | Numerical (string-encoded) |
| `seats` | Number of seats | Numerical |

> 🎯 **Target Variable → `selling_price`** (continuous, in Indian Rupees)  
> 📌 **Task Type → Regression**


---
## ⚙️ Phase 1 — Environment & Data Loading

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

TARGET = "selling_price"

### 1.1 — Load the Dataset

In [3]:
FILE_PATH = "cardekho_dataset.csv"

df = pd.read_csv(FILE_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 15,411 rows × 14 columns


### 1.2 — Initial Inspection

In [4]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(6)
memory usage: 2.3 MB


In [6]:
df.describe()

,Unnamed: 0,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price
count,15411.000000,15411.000000,1.541100e+04,15411.000000,15411.000000,15411.000000,15411.000000,1.541100e+04
mean,9811.857699,6.036338,5.561648e+04,19.701151,1486.057751,100.588254,5.325482,7.749711e+05
std,5643.418542,3.013291,5.161855e+04,4.171265,521.106696,42.972979,0.807628,8.941284e+05
min,0.000000,0.000000,1.000000e+02,4.000000,793.000000,38.400000,0.000000,4.000000e+04
25%,4906.500000,4.000000,3.000000e+04,17.000000,1197.000000,74.000000,5.000000,3.850000e+05
50%,9872.000000,6.000000,5.000000e+04,19.670000,1248.000000,88.500000,5.000000,5.560000e+05
75%,14668.500000,8.000000,7.000000e+04,22.700000,1582.000000,117.300000,5.000000,8.250000e+05
max,19543.000000,29.000000,3.800000e+06,33.540000,6592.000000,626.000000,9.000000,3.950000e+07


In [7]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})    .sort_values("Missing %", ascending=False)

,Missing Count,Missing %
Unnamed: 0,0,0.0
car_name,0,0.0
brand,0,0.0
model,0,0.0
vehicle_age,0,0.0
km_driven,0,0.0
seller_type,0,0.0
fuel_type,0,0.0
transmission_type,0,0.0
mileage,0,0.0


---
## 📊 Phase 2 — Exploratory Data Analysis (EDA)


### 2.1 — Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HINT: Plot selling_price distribution on axes[0] (raw) and axes[1] (log-transformed).
# - Library: matplotlib's .hist() works directly on a pandas Series.
# - For the log version, think about which numpy function undoes/applies log1p —
#   useful here because selling_price is heavily right-skewed (see df.describe() above).
# - Don't forget axes[0].set_title() / set_xlabel() so the plots are readable.

plt.tight_layout()
plt.show()

### 2.2 — Correlation Heatmap (Numerical Features)

In [ ]:
# HINT: You need three things —
# 1. A way to select only numeric columns from a DataFrame (pandas has a method for this).
# 2. A way to compute pairwise correlation between those columns (also a pandas method).
# 3. A way to visualize a matrix as a heatmap (seaborn is imported as sns — look for a
#    heatmap function; annot=True will print the values inside each cell).
# Question to answer once you've made the plot: which feature correlates most with selling_price?


### 2.3 — Selling Price vs Continuous Features (Scatter Plots)

In [ ]:
# HINT: Build a 2x2 grid with plt.subplots(2, 2, ...).
# - Note: this dataset does not have a 'year' column directly — check df.columns for
#   what's actually available (hint: it's an age-related column, not a year).
# - Loop over your 4 chosen features and axes together (zip is handy here) instead of
#   writing 4 near-identical blocks.
# - A scatter plot is the right chart type for two continuous variables — matplotlib's
#   .scatter() takes x, y, and an alpha for transparency (useful with 15k+ points).


### 2.4 — Categorical Features vs Selling Price (Box Plots)

In [ ]:
categorical_cols = ["fuel", "seller_type", "transmission", "owner"]
# NOTE: check df.columns — the actual column names in this dataset differ slightly
# from this list (e.g. it's fuel_type not fuel). Also, there is no 'owner' column here
# at all, so decide what to do about that entry.

# HINT: For each categorical column, you want a boxplot of selling_price grouped by
# that column's categories. Seaborn has a function built exactly for this (x=category,
# y=target). Loop over categorical_cols rather than copy-pasting per column.


### 2.5 — Car Age Analysis

In [ ]:
# HINT: 'car_age' isn't something you need to derive from scratch here — look closely
# at the columns already in df; one of them already represents age in years.
# Once you have it, a simple scatter plot (car_age vs selling_price) is enough to see
# the trend. Think about what relationship you'd expect (older -> cheaper) and whether
# the plot confirms it.


### 2.6 — Top Brands by Average Selling Price

In [ ]:
# HINT: This dataset already ships with a 'brand' column, so the string-splitting step
# from the original idea (extracting brand from a combined name column) isn't needed —
# check df.columns to confirm.
# - Group by brand and aggregate selling_price with a summary statistic (mean).
# - Sort the result and take the top 15.
# - For a ranked list of categories, a horizontal bar chart reads better than vertical
#   (pandas Series has a .plot(kind=...) shortcut).


---
## 🔧 Phase 3 — Data Preprocessing & Feature Engineering


In [ ]:
# Work on a copy — always preserve the original DataFrame
df_clean = df.copy()

### 3.1 — Drop Irrelevant / High-Cardinality Columns

In [ ]:
# HINT: Decide what to drop based on your own EDA above — think about:
# - Columns that just repeat information you've already turned into something more useful
#   (e.g. a raw identifier column, or a column you've derived a cleaner feature from).
# - Very high-cardinality text columns that won't help a tree/linear model directly.
# Use DataFrame.drop(columns=[...]) once you've decided your list.


### 3.2 — Handle Missing Values

In [ ]:
# This dataset has 0 missing values (confirmed in Phase 1), so there's technically
# nothing to impute here — but it's good practice to write defensive code anyway.
# HINT: pandas has methods to fill numeric columns with median and categorical columns
# with mode. Look up .fillna() and how to compute .median() / .mode() on a Series.
# Always re-check df_clean.isnull().sum() afterward to confirm you're at zero.


### 3.3 — Feature Engineering

In [ ]:
# HINT: You may already have car_age from EDA — reuse it here on df_clean instead of
# recomputing from a 'year' column that doesn't exist in this dataset.
# - This dataset's mileage/engine/max_power are already plain numbers (not strings like
#   "23.4 kmpl"), so the usual unit-stripping step (str.extract + regex) isn't needed here.
#   Check df_clean.dtypes to confirm before writing any extraction code.
# - Consider whether km_driven's distribution (see your EDA) would benefit from a log
#   transform — numpy has a function for this that pairs with expm1 later.


### 3.4 — Encode Categorical Variables

In [ ]:
# HINT: You need to turn categorical columns into numbers before modeling. Two options:
# - One-hot encoding: pandas has a function that does this directly on a DataFrame,
#   with a parameter to avoid the dummy-variable trap for nominal (unordered) categories.
# - Label/ordinal encoding: sklearn's LabelEncoder (already imported) fits better for
#   columns with a natural order.
# Decide which columns are nominal vs ordinal in this dataset and pick accordingly.


### 3.5 — (Optional) Target Transformation

In [ ]:
# HINT: Optional but often useful for skewed targets like selling_price.
# - numpy has a log transform function that handles zero values safely.
# - If you apply it here, remember: any predictions later need the inverse transform
#   applied before you compute real-money metrics like RMSE/MAE.


---
## 🤖 Phase 4 — Model Training & Evaluation


### 4.1 — Define Features (X) and Target (y)

In [ ]:
# HINT: Split df_clean into your feature matrix and target vector.
# - The target column is stored in the TARGET variable from Phase 1 — don't hardcode
#   the string again.
# - X = everything except target; y = just the target.


### 4.2 — Train / Test Split

In [ ]:
# HINT: sklearn's train_test_split (already imported) is what you need.
# - test_size: what fraction to hold out for testing (0.2 is a common default).
# - random_state: any fixed integer, so your split is reproducible on re-runs.


### 4.3 — Baseline Model: Linear Regression

In [ ]:
# HINT: Instantiate sklearn's LinearRegression, then look up the two method names
# every sklearn estimator uses to (1) learn from training data and (2) generate
# predictions on new data.


### 4.4 — Ensemble Model A: Random Forest Regressor

In [ ]:
# HINT: Instantiate RandomForestRegressor (imported already).
# Key parameters to think about:
# - n_estimators: how many trees (more = usually better but slower).
# - max_depth: None lets trees grow fully — powerful but can overfit on 15k rows.
# - random_state: for reproducibility.
# Then fit and predict, same pattern as the Linear Regression baseline.


### 4.5 — Ensemble Model B: Gradient Boosting Regressor

In [ ]:
# HINT: Instantiate GradientBoostingRegressor (imported already).
# Key parameters:
# - n_estimators, learning_rate: there's a tradeoff — lower learning_rate generally
#   needs more estimators to compensate.
# - max_depth: usually kept shallow (e.g. 3-5) for boosting, unlike Random Forest.
# Fit and predict the same way as your other two models.


### 4.6 — Evaluation Helper Function

In [ ]:
def evaluate_model(name: str, y_true, y_pred) -> dict:
    """Compute and display MSE, RMSE, MAE, and R² for a regression model."""
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)

    print(f"\n{'─'*42}")
    print(f"  Model : {name}")
    print(f"{'─'*42}")
    print(f"  MSE   : {mse:>15,.2f}")
    print(f"  RMSE  : {rmse:>15,.2f}   ← same unit as target (INR)")
    print(f"  MAE   : {mae:>15,.2f}")
    print(f"  R²    : {r2:>15.4f}   ← 1.0 = perfect")
    return {"Model": name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

# TODO: Call evaluate_model() for each trained model.
# results = []
# results.append(evaluate_model("Linear Regression", y_test, lr_preds))
# results.append(evaluate_model("Random Forest",     y_test, rf_preds))
# results.append(evaluate_model("Gradient Boosting", y_test, gb_preds))

### 4.7 — Model Comparison Table

In [ ]:
# HINT: You should have a list of dicts (one per model) from calling evaluate_model()
# three times above. Turn that list into a DataFrame, set the model name as the index,
# and sort by whichever metric tells you "higher is better" for regression.


### 4.8 — Feature Importance (Random Forest)

In [ ]:
# HINT: A fitted RandomForestRegressor exposes an attribute with one importance score
# per feature (check the sklearn docs / autocomplete on rf_model for the exact name).
# - Pair those scores with your feature names (X_train.columns) using a pandas Series.
# - Sort and take the top 15, then a horizontal bar plot works well for ranked importance.


### 4.9 — Residual Plot

In [ ]:
# HINT: Residuals = actual - predicted. Compute this for your best-performing model's
# test predictions.
# - Scatter plot: predicted values on x-axis, residuals on y-axis.
# - Draw a horizontal reference line at 0 (matplotlib's axhline) — a well-fit model
#   should show residuals scattered randomly around that line with no obvious pattern
#   (a funnel or curve shape would signal a problem).


### 4.10 — Cross-Validation (Recommended)

In [ ]:
# HINT: sklearn's cross_val_score (already imported) does this in one call.
# - Pass the full X and y (not the train split) — CV handles the splitting internally.
# - cv=5 for 5-fold, scoring="r2".
# - Report both the mean and the standard deviation of the fold scores — std tells you
#   how stable the model's performance is across different subsets of data.


---
✅ **Blueprint complete!** Fill in all `TODO` blocks above, run each cell, and interpret your results.
